In [ ]:
!pip install -q huggingface_hub
!pip install -U "transformers>=4.40.0"


from huggingface_hub import login

# Option 1: Hard-code your token (not recommended to share)
# token = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX"
# login(token=token)

# Option 2: Enter securely at runtime
token = input("Paste your Hugging Face access token: ").strip()
login(token=token)
print("Logged in to Hugging Face Hub.")


In [ ]:
# ============================================================================
# PEFT / LoRA FINE-TUNING — MULTILINGUAL TEXT CLASSIFICATION
# ALL 3 MODELS × ALL 3 LANGUAGES
#
# Models  : xlm-roberta-base | ai4bharat/indic-bert | microsoft/mdeberta-v3-base
# Languages: Hindi | Tamil | Telugu
# Metrics : Accuracy, F1 Macro, per-class Precision / Recall / F1
# ============================================================================

# INSTALLS (uncomment if needed in Colab)
# !pip install transformers peft accelerate datasets scikit-learn openpyxl tqdm -q

# ============================================================================
# SECTION 1 — IMPORTS
# ============================================================================
import os
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)
from huggingface_hub import login

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)


warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "/content/hf_cache"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"])
    print("✓ Logged in to Hugging Face Hub.")
else:
    print("⚠️  HF_TOKEN not set. Gated models may not be accessible.")



# ============================================================================
# SECTION 2 — REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
# ============================================================================
# SECTION 3 — CONFIGURATION
# ============================================================================

# --- Datasets (one per language) ---
DATASETS = {
    'Hindi' : '/content/drive/MyDrive/Datasets/Hindi_dataset/hindi_filtered_new_criteria.xlsx',
    'Tamil' : '/content/drive/MyDrive/Datasets/Tamil_dataset/filtered_dataset_tamil.xlsx',
    'Telugu': '/content/drive/MyDrive/Datasets/Telugu_dataset/filtered_dataset_telugu.xlsx',
}

# --- Models with architecture-specific LoRA target modules ---
MODELS = {
    'mdeberta': {
        'model_id'       : 'microsoft/mdeberta-v3-base',
        'target_modules' : ['query_proj', 'value_proj'],            # DeBERTa attention
    },

    'indicbert': {
        'model_id'       : 'ai4bharat/indic-bert',
        'target_modules' : ['query', 'key', 'value', 'dense'],      # ALBERT shared layers
    },
    'xlm-roberta': {
        'model_id'       : 'xlm-roberta-base',
        'target_modules' : ['query', 'value'],                      # RoBERTa attention
    },


}

TEXT_COL        = 'text'
LABEL_COL       = 'labels'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/Models'

# --- Hyperparameter grid ---
# Shifted upward — lower LRs cause collapse on IndicBERT/mDeBERTa
BATCH_SIZES    = [32, 64]
LEARNING_RATES = [3e-4, 5e-4, 7e-4]

# --- Training config ---
MAX_LEN        = 128
NUM_EPOCHS     = 5
GAMMA          = 2.5    # Focal Loss focusing parameter
MINORITY_BOOST = 1.0    # Set to 1.0 — Focal Loss alone handles imbalance
                        # (combining both caused gradient collapse on small models)
TARGET_RECALL  = 0.95   # Class 1 recall target for early stopping

# --- Train / Val / Test split ---
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# --- LoRA config ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.1
# ============================================================================
# SECTION 4 — FOCAL LOSS
# ============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss — down-weights easy examples, focuses on hard minority samples.
    alpha  : per-class weights tensor
    gamma  : focusing parameter (higher = more focus on hard examples)
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# ============================================================================
# SECTION 5 — CUSTOM CALLBACK: RECALL EARLY STOPPING
# ============================================================================
class RecallEarlyStopping(TrainerCallback):
    """Stop training as soon as Class 1 recall reaches TARGET_RECALL."""

    def __init__(self, target_recall=0.95, patience=2):
        super().__init__()
        self.target_recall    = target_recall
        self.patience         = patience
        self.patience_counter = 0
        self.best_recall      = 0.0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control

        recall_class_1 = metrics.get("eval_recall_class_1", 0)

        if recall_class_1 >= self.target_recall:
            control.should_training_stop = True
            control.should_save          = True

        if recall_class_1 > self.best_recall:
            self.best_recall      = recall_class_1
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
        return control


# ============================================================================
# SECTION 6 — FOCAL LOSS TRAINER
# ============================================================================
class FocalLossTrainer(Trainer):
    """HuggingFace Trainer that replaces cross-entropy with Focal Loss."""

    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(
                class_weights, dtype=torch.float
            ).to(self.args.device)
        else:
            self.class_weights = None
        self.gamma = gamma
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fct = FocalLoss(alpha=self.class_weights, gamma=self.gamma)
        loss     = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


# ============================================================================
# SECTION 7 — DATASET LOADING & STRATIFIED SPLIT
# ============================================================================
def load_and_split(
    file_path,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_state=42,
    save_dir='./splits',
):
    print("=" * 60)

    if file_path.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_path)
    elif file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported format: {file_path}")

    df = df[[text_col, label_col]].dropna().reset_index(drop=True)

    print(f"\n📊 Original dataset : {len(df):,} rows")
    class_counts = df[label_col].value_counts().sort_index()
    for lbl, cnt in class_counts.items():
        print(f"   Class {lbl}: {cnt:,}  ({cnt/len(df)*100:.1f}%)")

    imbalance = class_counts.max() / class_counts.min()
    if imbalance > 3:
    # Stratified splits
    train_val, test_df = train_test_split(
        df, test_size=test_ratio, random_state=random_state, stratify=df[label_col]
    )
    adjusted_val = val_ratio / (train_ratio + val_ratio)
    train_df, val_df = train_test_split(
        train_val, test_size=adjusted_val, random_state=random_state,
        stratify=train_val[label_col]
    )

    for name, split in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
        counts = split[label_col].value_counts().sort_index()
        print(f"\n   {name} ({len(split):,}):")
        for lbl, cnt in counts.items():
            print(f"      Class {lbl}: {cnt:,}  ({cnt/len(split)*100:.1f}%)")

    # Class weights with configurable minority boost
    labels_arr = train_df[label_col].values
    unique_cls = np.unique(labels_arr)
    n          = len(labels_arr)
    weights    = [n / (len(unique_cls) * np.sum(labels_arr == c)) for c in sorted(unique_cls)]
    weights    = [w / min(weights) for w in weights]
    min_idx    = int(np.argmin([np.sum(labels_arr == c) for c in sorted(unique_cls)]))
    weights[min_idx] *= MINORITY_BOOST
    os.makedirs(save_dir, exist_ok=True)
    train_df.to_csv(f"{save_dir}/train.csv", index=False)
    val_df.to_csv(f"{save_dir}/val.csv",     index=False)
    test_df.to_csv(f"{save_dir}/test.csv",   index=False)
    with open(f"{save_dir}/class_weights.json", "w") as f:
        json.dump({"class_weights": weights}, f, indent=2)
    return train_df, val_df, test_df, weights


# ============================================================================
# SECTION 8 — TOKENIZATION HELPER
# ============================================================================
def tokenize_datasets(train_df, val_df, test_df, tokenizer, text_col, label_col):
    """Tokenize all three splits and return HF Dataset objects."""

    def df_to_hf(df):
        return Dataset.from_dict({
            'text' : df[text_col].tolist(),
            'label': df[label_col].tolist(),
        })

    def tokenize_fn(examples):
        return tokenizer(
            examples['text'],
            padding    = 'max_length',
            truncation = True,
            max_length = MAX_LEN,
        )
    train_ds = df_to_hf(train_df).map(tokenize_fn, batched=True).remove_columns(['text'])
    val_ds   = df_to_hf(val_df).map(tokenize_fn,   batched=True).remove_columns(['text'])
    test_ds  = df_to_hf(test_df).map(tokenize_fn,  batched=True).remove_columns(['text'])

    for ds in [train_ds, val_ds, test_ds]:
        ds.set_format('torch')

    print(f"   ✅ Train : {len(train_ds):,}   Val : {len(val_ds):,}   Test : {len(test_ds):,}")
    return train_ds, val_ds, test_ds


# ============================================================================
# SECTION 9 — METRICS
# ============================================================================
def compute_metrics(eval_pred):
    """
    Accuracy, F1 Macro, and per-class Precision / Recall / F1.
    recall_class_1 is monitored by RecallEarlyStopping.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1]
    )

    return {
        'accuracy'          : accuracy,
        'f1_macro'          : f1_macro,
        'precision_class_0' : precision[0],
        'recall_class_0'    : recall[0],
        'f1_class_0'        : f1[0],
        'precision_class_1' : precision[1],
        'recall_class_1'    : recall[1],   # ← used by RecallEarlyStopping
        'f1_class_1'        : f1[1],
    }


# ============================================================================
# SECTION 10 — BUILD LORA MODEL
# ============================================================================
def build_lora_model(model_id, target_modules, num_labels=2):
    """
    Load a base classification model and wrap it with LoRA adapters.
    target_modules must match the architecture:
      - xlm-roberta : ['query', 'value']
      - indic-bert  : ['query', 'key', 'value', 'dense']  (ALBERT shared layers)
      - mdeberta    : ['query_proj', 'value_proj']
    """
    hf_token   = os.environ.get("HF_TOKEN")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=num_labels, token=hf_token
    )

    lora_cfg = LoraConfig(
        task_type      = TaskType.SEQ_CLS,
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        bias           = 'none',
        target_modules = target_modules,
    )

    # Build once and print — do NOT call get_peft_model twice
    model = get_peft_model(base_model, lora_cfg)
    return model


# ============================================================================
# SECTION 11 — THRESHOLD ADJUSTMENT
# ============================================================================
def find_and_apply_optimal_threshold(trainer, val_dataset, test_dataset,
                                     target_recall=0.95):
    """
    Find the lowest threshold on the validation set that meets target recall,
    then evaluate the test set at that threshold.
    """
    print("\n" + "=" * 60)
    print("🎯 THRESHOLD OPTIMIZATION FOR 95% RECALL")
    print("=" * 60)

    val_out    = trainer.predict(val_dataset)
    val_probs  = torch.softmax(torch.tensor(val_out.predictions), dim=1)[:, 1].numpy()
    val_labels = val_out.label_ids

    precisions, recalls, thresholds = precision_recall_curve(val_labels, val_probs, pos_label=1)
    idx = np.where(recalls >= target_recall)[0]

    if len(idx) == 0:
        print(f"❌ Cannot reach {target_recall*100:.0f}% recall on validation set.")
        return None

    best_idx = idx[np.argmax(precisions[idx])]
    opt_thr  = thresholds[best_idx]

    test_out    = trainer.predict(test_dataset)
    test_probs  = torch.softmax(torch.tensor(test_out.predictions), dim=1)[:, 1].numpy()
    test_labels = test_out.label_ids
    test_preds  = (test_probs >= opt_thr).astype(int)

    acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average=None, labels=[0, 1]
    )

    print(f"\n📊 Optimal threshold : {opt_thr:.4f}")
    print(f"   Accuracy          : {acc:.4f}")
    print(f"   Class 0 — Prec: {precision[0]:.4f}  Rec: {recall[0]:.4f}  F1: {f1[0]:.4f}")
    print(f"   Class 1 — Prec: {precision[1]:.4f}  Rec: {recall[1]:.4f}  "
          f"F1: {f1[1]:.4f}  "
          f"{'✅ MEETS TARGET!' if recall[1] >= target_recall else '⚠️ Below target'}")

    cm = confusion_matrix(test_labels, test_preds)
    print(f"\n   Confusion Matrix:\n{cm}")
    print(f"   TN:{cm[0,0]}  FP:{cm[0,1]}  FN:{cm[1,0]}  TP:{cm[1,1]}")
    print("\n" + classification_report(test_labels, test_preds, digits=4))

    return {
        'threshold'        : float(opt_thr),
        'accuracy'         : acc,
        'precision_class_0': precision[0], 'recall_class_0': recall[0], 'f1_class_0': f1[0],
        'precision_class_1': precision[1], 'recall_class_1': recall[1], 'f1_class_1': f1[1],
        'confusion_matrix' : cm,
    }


# ============================================================================
# SECTION 12 — SINGLE TRAINING RUN
# ============================================================================
def run_single_training(
    train_ds, val_ds, test_ds,
    class_weights,
    lr, batch_size,
    run_output_dir,
    model_id,
    target_modules,
):
    """Build a fresh LoRA model, train it, evaluate on test set, return results."""
    # Fresh model for every hyperparameter combination
    model = build_lora_model(model_id, target_modules, num_labels=2)
    is_mdeberta = 'deberta' in model_id.lower()

    training_args = TrainingArguments(
        output_dir                  = run_output_dir,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        learning_rate               = lr,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        num_train_epochs            = NUM_EPOCHS,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        logging_steps               = 10,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'recall_class_1',
        greater_is_better           = True,
        report_to                   = 'none',
        #fp16                        = torch.cuda.is_available(),
        fp16 = torch.cuda.is_available() and not is_mdeberta,
        bf16 = torch.cuda.is_available() and is_mdeberta,
    )

    hf_token  = os.environ.get("HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

    trainer = FocalLossTrainer(
        model            = model,
        args             = training_args,
        train_dataset    = train_ds,
        eval_dataset     = val_ds,
        processing_class = tokenizer,       # replaces deprecated 'tokenizer='
        compute_metrics  = compute_metrics,
        class_weights    = class_weights,
        gamma            = GAMMA,
        callbacks        = [
            RecallEarlyStopping(target_recall=TARGET_RECALL, patience=2),
        ],
    )

    train_result = trainer.train()

    # Evaluate on test set
    pred_output = trainer.predict(test_ds)
    logits      = pred_output.predictions
    labels      = pred_output.label_ids
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1]
    )
    print("\n" + "-" * 60)
    print(classification_report(labels, predictions, digits=4))

    return {
        'lr'                : lr,
        'batch_size'        : batch_size,
        'accuracy'          : accuracy,
        'f1_macro'          : f1_macro,
        'precision_class_0' : float(precision[0]),
        'recall_class_0'    : float(recall[0]),
        'f1_class_0'        : float(f1[0]),
        'precision_class_1' : float(precision[1]),
        'recall_class_1'    : float(recall[1]),
        'f1_class_1'        : float(f1[1]),
        'trainer'           : trainer,
        'val_ds'            : val_ds,
        'test_ds'           : test_ds,
    }


# ============================================================================
# SECTION 13 — SINGLE LANGUAGE × MODEL PIPELINE
# ============================================================================
def run_experiment(lang, model_key, train_ds, val_ds, test_ds, class_weights):
    """
    Run the full grid search for one language + model combination.
    Returns list of all run results and the best run dict.
    """
    model_cfg      = MODELS[model_key]
    model_id       = model_cfg['model_id']
    target_modules = model_cfg['target_modules']
    output_dir     = os.path.join(BASE_OUTPUT_DIR, f'{lang}_{model_key}_lora')
    all_results = []
    best_recall = -1.0
    best_run    = None

    for lr in LEARNING_RATES:
        for bs in BATCH_SIZES:
            run_dir = os.path.join(output_dir, f'tmp_lr{lr}_bs{bs}')
            os.makedirs(run_dir, exist_ok=True)

            results = run_single_training(
                train_ds       = train_ds,
                val_ds         = val_ds,
                test_ds        = test_ds,
                class_weights  = class_weights,
                lr             = lr,
                batch_size     = bs,
                run_output_dir = run_dir,
                model_id       = model_id,
                target_modules = target_modules,
            )
            all_results.append(results)

            if results['recall_class_1'] > best_recall:
                best_recall = results['recall_class_1']
                best_run    = results

    # Threshold tuning if best recall is close but not yet at target
    if best_run['recall_class_1'] < TARGET_RECALL and best_run['recall_class_1'] >= 0.90:
        print("\n🔧 Attempting threshold adjustment to reach 95% recall...")
        tuned = find_and_apply_optimal_threshold(
            best_run['trainer'],
            best_run['val_ds'],
            best_run['test_ds'],
            target_recall=TARGET_RECALL,
        )
        if tuned:
            best_run.update({
                'recall_class_1'    : tuned['recall_class_1'],
                'precision_class_1' : tuned['precision_class_1'],
                'f1_class_1'        : tuned['f1_class_1'],
                'threshold'         : tuned['threshold'],
            })

    # Print grid search summary
    print("\n" + "=" * 60)
    print(f"📊 GRID SEARCH SUMMARY — {lang.upper()} / {model_key.upper()}")
    print("=" * 60)
    rows = [{
        'LR'            : r['lr'],
        'Batch Size'    : r['batch_size'],
        'Accuracy'      : f"{r['accuracy']:.4f}",
        'F1 Macro'      : f"{r['f1_macro']:.4f}",
        'Recall (C1)'   : f"{r['recall_class_1']:.4f}",
        'F1 (C1)'       : f"{r['f1_class_1']:.4f}",
        'Precision (C1)': f"{r['precision_class_1']:.4f}",
    } for r in all_results]
    print(pd.DataFrame(rows).to_string(index=False))

    # Save CSV
    csv_path = f'lora_results_{lang}_{model_key}.csv'
    if 'threshold' in best_run:
        print(f"   Applied threshold : {best_run['threshold']:.4f}")

    # Save best model
    save_dir = os.path.join(output_dir, f'{lang}_{model_key}_best')
    os.makedirs(save_dir, exist_ok=True)
    best_run['trainer'].model.save_pretrained(save_dir)
    hf_token  = os.environ.get("HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    tokenizer.save_pretrained(save_dir)

    meta = {
        'language'       : lang,
        'model_key'      : model_key,
        'model_id'       : model_id,
        'lora_r'         : LORA_R,
        'lora_alpha'     : LORA_ALPHA,
        'gamma'          : GAMMA,
        'minority_boost' : MINORITY_BOOST,
        'class_weights'  : class_weights,
        'best_lr'        : best_run['lr'],
        'best_batch_size': best_run['batch_size'],
        'accuracy'       : best_run['accuracy'],
        'f1_macro'       : best_run['f1_macro'],
        'recall_class_1' : best_run['recall_class_1'],
        'f1_class_1'     : best_run['f1_class_1'],
    }
    if 'threshold' in best_run:
        meta['threshold'] = best_run['threshold']

    with open(f"{save_dir}/lora_meta.json", 'w') as f:
        json.dump(meta, f, indent=2)
    return all_results, best_run


# ============================================================================
# SECTION 14 — RUN ALL LANGUAGES × ALL MODELS
# ============================================================================
if __name__ == '__main__':
    from google.colab import drive
    drive.mount('/content/drive')

    # Master results table across all 9 experiments
    master_rows    = []
    all_experiment_results = {}

    for lang, dataset_path in DATASETS.items():

        if not os.path.exists(dataset_path):
            print(f"\n⚠️  Dataset not found for {lang}: {dataset_path} — skipping.")
            continue
        # Load & split ONCE per language (shared across all 3 models)
        train_df, val_df, test_df, class_weights = load_and_split(
            file_path = dataset_path,
            text_col  = TEXT_COL,
            label_col = LABEL_COL,
            save_dir  = os.path.join(BASE_OUTPUT_DIR, f'splits_{lang.lower()}'),
        )

        for model_key, model_cfg in MODELS.items():
            model_id  = model_cfg['model_id']
            hf_token  = os.environ.get("HF_TOKEN")

            # Tokenize fresh for each model (different tokenizers)
            tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
            train_ds, val_ds, test_ds = tokenize_datasets(
                train_df, val_df, test_df, tokenizer, TEXT_COL, LABEL_COL
            )

            # Run grid search for this language × model
            all_results, best_run = run_experiment(
                lang          = lang,
                model_key     = model_key,
                train_ds      = train_ds,
                val_ds        = val_ds,
                test_ds       = test_ds,
                class_weights = class_weights,
            )

            all_experiment_results[f'{lang}_{model_key}'] = best_run

            # Append to master results
            for r in all_results:
                master_rows.append({
                    'Language'      : lang,
                    'Model'         : model_key,
                    'LR'            : r['lr'],
                    'Batch Size'    : r['batch_size'],
                    'Accuracy'      : round(r['accuracy'],    4),
                    'F1 Macro'      : round(r['f1_macro'],    4),
                    'Precision C0'  : round(r['precision_class_0'], 4),
                    'Recall C0'     : round(r['recall_class_0'],    4),
                    'F1 C0'         : round(r['f1_class_0'],        4),
                    'Precision C1'  : round(r['precision_class_1'], 4),
                    'Recall C1'     : round(r['recall_class_1'],    4),
                    'F1 C1'         : round(r['f1_class_1'],        4),
                    'Meets Target'  : '✅' if r['recall_class_1'] >= TARGET_RECALL else '⚠️',
                })

    # Save master CSV covering all 9 experiments
    master_df = pd.DataFrame(master_rows)
    master_csv = os.path.join(BASE_OUTPUT_DIR, 'lora_all_results_master.csv')
    master_df.to_csv(master_csv, index=False)
    print(f"\n💾 Master results saved to: {master_csv}")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
⚠️  HF_TOKEN not set. Gated models may not be accessible.
✓ All imports successful.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  LOADING DATASET FOR LANGUAGE: Hindi

📂 DATASET LOADING WITH STRATIFICATION

📊 Original dataset : 4,037 rows
   Class 0: 3,559  (88.2%)
   Class 1: 478  (11.8%)

   ⚠️  Imbalance ratio: 7.45:1  → Focal Loss will help

   TRAIN (2,825):
      Class 0: 2,491  (88.2%)
      Class 1: 334  (11.8%)

   VAL (606):
      Class 0: 534  (88.1%)
      Class 1: 72  (11.9%)

   TEST (606):
      Class 0: 534  (88.1%)
      Class 1: 72  (11.9%)

📊 Class weights (minority boost=1.0×): [np.float64(1.0), np.float64(7.45808383233533)]
💾 Splits saved to /content/drive/MyDrive/Models/splits_hindi/


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]


🔄 Tokenizing datasets...


...

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 1,181,954 || all params: 279,227,140 || trainable%: 0.4233
   📊 FocalLoss — gamma=2.5, class_weights=[np.float64(1.0), np.float64(5.263063063063063)]
🚀 Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.252730,0.149655,0.966443,0.937707,0.980800,0.979233,0.980016,0.891667,0.899160,0.895397
2,0.172113,0.084914,0.944966,0.908064,0.994924,0.939297,0.966311,0.753247,0.974790,0.849817



🎯 Target recall 95.00% achieved! Current: 0.9748
   Stopping training early...
   ✅ Training done — loss: 0.2731



📊 Test Set Results:
   Accuracy   : 0.9302
   F1 Macro   : 0.8857
   Class 0 — Prec: 0.9914  Rec: 0.9249  F1: 0.9570
   Class 1 — Prec: 0.7081  Rec: 0.9580  F1: 0.8143
   ✅ Class 1 recall 0.9580 meets target!

------------------------------------------------------------
              precision    recall  f1-score   support

           0     0.9914    0.9249    0.9570       626
           1     0.7081    0.9580    0.8143       119

    accuracy                         0.9302       745
   macro avg     0.8498    0.9415    0.8857       745
weighted avg     0.9462    0.9302    0.9342       745


📊 GRID SEARCH SUMMARY — TELUGU / XLM-ROBERTA
    LR  Batch Size Accuracy F1 Macro Recall (C1) F1 (C1) Precision (C1)
0.0003          32   0.8805   0.8233      0.9748  0.7227         0.5743
0.0003          64   0.8711   0.8124      0.9748  0.7073         0.5550
0.0005          32   0.9181   0.8691      0.9580  0.7889         0.6706
0.0005          64   0.9436   0.9050      0.9580  0.8444         0.